# Task 8: Machine Learning Model

Intern: [Raj Bhut]## 1. Introduction

The objective of this task is to build a machine learning model that predicts house prices using property features such as area, bedrooms, bathrooms, floors, year built, location, condition, and garage availability.

- Data preprocessing
- Feature selection
- Train/test split
- Model training
- Evaluation using RMSE

## 2. Setup: Import Libraries

This section imports required Python libraries, sets display options, and defines the `DATA_PATH`. Run the setup cell before running data loading and model training cells.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
print("Libraries imported successfully!")

Libraries imported successfully!


## 3. Data Loading and Inspection

In this section, the dataset is loaded from the local CSV file and inspected for basic quality checks.

The inspection includes:
- confirming the dataset shape
- listing available columns
- checking for missing values
- reviewing summary statistics
- preparing the target and feature columns for model building

In [2]:
DATA_PATH = Path(r"C:\Users\Admin\Downloads\House Price Prediction Dataset.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

df = pd.read_csv(DATA_PATH, encoding="latin1")
df.head()


,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price
0,1,1360,5,4,3,1970,Downtown,Excellent,No,149919
1,2,4272,5,4,3,1958,Downtown,Excellent,No,424998
2,3,3592,2,2,3,1938,Downtown,Good,No,266746
3,4,966,4,2,2,1902,Suburban,Fair,Yes,244020
4,5,4926,1,4,2,1975,Downtown,Fair,Yes,636056


In [3]:
# Normalize column names
df.columns = df.columns.str.strip()

# Basic sanity checks
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum())

print("\nBasic statistics:")
display(df.describe(include="all"))

Dataset shape: (2000, 10)

Columns:
['Id', 'Area', 'Bedrooms', 'Bathrooms', 'Floors', 'YearBuilt', 'Location', 'Condition', 'Garage', 'Price']

Missing values:


Id           0
Area         0
Bedrooms     0
Bathrooms    0
Floors       0
YearBuilt    0
Location     0
Condition    0
Garage       0
Price        0
dtype: int64


Basic statistics:


,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price
count,2000.000000,2000.000000,2000.000000,2000.00000,2000.000000,2000.000000,2000,2000,2000,2000.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,4,4,2,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,Downtown,Fair,No,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,558,521,1038,NaN
mean,1000.500000,2786.209500,3.003500,2.55250,1.993500,1961.446000,NaN,NaN,NaN,537676.855000
std,577.494589,1295.146799,1.424606,1.10899,0.809188,35.926695,NaN,NaN,NaN,276428.845719
min,1.000000,501.000000,1.000000,1.00000,1.000000,1900.000000,NaN,NaN,NaN,50005.000000
25%,500.750000,1653.000000,2.000000,2.00000,1.000000,1930.000000,NaN,NaN,NaN,300098.000000
50%,1000.500000,2833.000000,3.000000,3.00000,2.000000,1961.000000,NaN,NaN,NaN,539254.000000
75%,1500.250000,3887.500000,4.000000,4.00000,3.000000,1993.000000,NaN,NaN,NaN,780086.000000


## Feature Selection

The target column is `Price`. The `Id` column is not useful for prediction, so it is removed from model features.

In [4]:
target = "Price"
drop_columns = ["Id", target]

X = df.drop(columns=drop_columns)
y = df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['Area', 'Bedrooms', 'Bathrooms', 'Floors', 'YearBuilt']
Categorical features: ['Location', 'Condition', 'Garage']


C:\Users\Admin\AppData\Local\Temp\ipykernel_23444\2734375259.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


## Model Training

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42)
}

results = []
trained_models = {}

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    
    rmse = root_mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    
    results.append({
        "Model": model_name,
        "RMSE": rmse,
        "R2 Score": r2
    })
    trained_models[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df

,Model,RMSE,R2 Score
0,Linear Regression,279859.725838,-0.006718
1,Random Forest,293288.858887,-0.105651


## Prediction Example

In [6]:
sample_house = X_test.head(1)
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

predicted_price = best_model.predict(sample_house)[0]

print("Best model:", best_model_name)
print("Predicted price:", round(predicted_price, 2))
print("Actual price:", y_test.iloc[0])
display(sample_house)

Best model: Linear Regression
Predicted price: 521988.22
Actual price: 514764


,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage
1860,633,1,4,2,1901,Urban,Fair,No


## Save Model

In [7]:
import pickle

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

with open("task8_house_price_model_raj_bhut.pkl", "wb") as file:
    pickle.dump(best_model, file)

print("Saved model: task8_house_price_model_raj_bhut.pkl")

Saved model: task8_house_price_model_raj_bhut.pkl


## Conclusion

The house price prediction model was built successfully using both numeric and categorical features from the dataset. Linear Regression provided a simple baseline, while Random Forest was included to capture more complex relationships between property attributes and price.

The evaluation results show the model performance using RMSE and R2 Score, which helps compare predictions against actual prices. Overall, this notebook demonstrates a complete workflow from data loading and inspection to model training, prediction, and saving the final model for reuse.